# L85 · KV-Cache 从零实现 — 键值缓存矩阵更新，O(seq²)→O(seq) 加速

**学习目标**
- 理解 KV-Cache 解决了 Transformer 推理的哪个瓶颈
- 实现 scaled dot-product attention（纯 NumPy）
- 实现 KVCache 类：append + retrieve
- 比较有 / 无 KV 缓存的推理速度

← **上一课**　[L84 · LoRA 低秩适配](L84_lora.ipynb)

> 上节课学习了 **LoRA 低秩适配**：W = W₀ + BA，用 0.1% 参数量精调 GPT-style 模型。  
> 本课将探讨 **KV-Cache 从零实现**。

## 为什么需要 KV-Cache？

自回归生成（Autoregressive Generation，AR）：第 t 步需要对 token 0..t 做 attention。

**无缓存**：每步重新计算所有 K, V 投影 → O(t²) work per step → O(T³) total
**有缓存**：存储所有过去的 K, V，每步只计算新 token 的 K, V，append 到缓存 → O(t) work per step → O(T²) total

两者 total FLOPs 量级相同，但缓存版避免了~T倍的重复投影计算，实际速度快 T 倍左右。

In [ ]:
import numpy as np
import time

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention，纯 NumPy。

    Args:
        Q: (n_heads, seq_q, head_dim)
        K: (n_heads, seq_k, head_dim)
        V: (n_heads, seq_k, head_dim)
        mask: optional (seq_q, seq_k) bool mask，True=mask out

    Returns:
        (n_heads, seq_q, head_dim)
    """
    # ✏️ TODO: scores = Q @ K^T / sqrt(head_dim)
    raise NotImplementedError("TODO")

# 验证
np.random.seed(0)
n_heads, seq, d = 2, 4, 8
Q = np.random.randn(n_heads, seq, d)
K = np.random.randn(n_heads, seq, d)
V = np.random.randn(n_heads, seq, d)
out = scaled_dot_product_attention(Q, K, V)
assert out.shape == (n_heads, seq, d)
print(f'attention 输出 shape: {out.shape}  ✅')

## ✏️ 实现 KVCache

In [ ]:
class KVCache:
    """KV 缓存：每个 Transformer 层存储历史 K, V 并逐步 append。"""

    def __init__(self, n_heads: int, head_dim: int):
        self.n_heads = n_heads
        self.head_dim = head_dim
        self._k = {}   # layer → (n_heads, seq, head_dim)
        self._v = {}

    def update(self, layer, new_k, new_v):
        """Append 新 K/V 并返回完整缓存。new_k/new_v: (n_heads, 1, head_dim)"""
        # ✏️ TODO: concatenate 到 axis=1
        raise NotImplementedError("TODO")

    def seq_len(self):
        if not self._k: return 0
        return next(iter(self._k.values())).shape[1]

# 也可以直接从 aurora.llm 导入：
# from aurora.llm import KVCache

In [ ]:
# 验证：有缓存 vs 无缓存的 attention 输出必须完全一致（因果掩码对齐）
n_heads, total_seq, d = 2, 6, 8
np.random.seed(42)

K_all = np.random.randn(n_heads, total_seq, d)
V_all = np.random.randn(n_heads, total_seq, d)
Q_all = np.random.randn(n_heads, total_seq, d)

# 无缓存：带因果掩码的全量计算
# mask[i, j]=True 表示 mask out（屏蔽未来位置 j > i）
causal_mask = ~np.tril(np.ones((total_seq, total_seq), dtype=bool))

try:
    out_no_cache = scaled_dot_product_attention(Q_all, K_all, V_all, mask=causal_mask)

    # 有缓存：逐 token 追加，参数与测试数据对齐
    cache = KVCache(n_heads=n_heads, head_dim=d)
    out_with_cache = np.zeros_like(Q_all)
    for t in range(total_seq):
        new_k = K_all[:, t:t+1, :]
        new_v = V_all[:, t:t+1, :]
        new_q = Q_all[:, t:t+1, :]
        k_full, v_full = cache.update(0, new_k, new_v)
        out_t = scaled_dot_product_attention(new_q, k_full, v_full)
        out_with_cache[:, t:t+1, :] = out_t

    # 对比：两者的每个 token 输出应完全一致
    assert np.allclose(out_no_cache, out_with_cache, atol=1e-9), (
        f'有无缓存结果不一致！\n'
        f'max diff = {np.abs(out_no_cache - out_with_cache).max():.2e}\n'
        f'提示：检查 scaled_dot_product_attention 的因果掩码逻辑'
    )
    print(f'无缓存输出 shape: {out_no_cache.shape}')
    print(f'有缓存输出 shape: {out_with_cache.shape}')
    print(f'缓存长度（token 数）: {cache.seq_len()}')
    print('✅ 有缓存 == 无缓存（因果掩码对齐），KV-Cache 实现正确')

except NotImplementedError:
    print('⬜ scaled_dot_product_attention 或 KVCache.update 未实现')


## 速度对比

In [ ]:
# 演示：朴素 O(T²) 投影 vs 缓存 O(T) 投影
print(f'{"T":>4}  {"无缓存(ms)":>11}  {"有缓存(ms)":>11}  {"加速比":>6}')
print('-' * 40)

for T in [10, 50, 100, 200]:
    d_model, n_heads_t, head_dim = 64, 4, 16
    W_k = np.random.randn(d_model, d_model)

    # 无缓存：每步重新投影所有 T 个 token
    t0 = time.perf_counter()
    for step in range(T):
        all_x = np.random.randn(step+1, d_model)
        _ = all_x @ W_k
    no_cache_ms = (time.perf_counter() - t0) * 1000

    # 有缓存：每步只投影 1 个新 token
    t0 = time.perf_counter()
    for step in range(T):
        new_x = np.random.randn(1, d_model)
        _ = new_x @ W_k
    cache_ms = (time.perf_counter() - t0) * 1000

    speedup = no_cache_ms / max(cache_ms, 1e-6)
    print(f'{T:>4}  {no_cache_ms:>11.2f}  {cache_ms:>11.2f}  {speedup:>6.1f}×')

## 本课收束

| 概念 | 要记住的 |
|---|---|
| KV-Cache | 存储过去的 K/V，每步 append 一个新 slice |
| 加速本质 | 避免了 T 倍重复的 K/V 线性投影 |
| 内存代价 | 缓存大小 O(T × n_layers × n_heads × head_dim) |
| L86 | 采样策略——有了 logits，如何选 token？ |

下一课（L86）实现采样策略：top-k / top-p / temperature 控制 LLM 的输出多样性，这是 KV 缓存之上的解码层。

## ✏️ 闭卷推导检查格 — KV-Cache 代价分析

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：推理时生成长度为 $T$ 的序列，模型维度 $d$。

填写下表（每行填一步的计算代价，用大 O 表示）：

| | 无缓存（每步重算所有 token）| 有缓存（只计算新 token）|
|---|---|---|
| 第 $t$ 步：K/V 投影代价 | $O(t \cdot d^2)$ | $O(___)$ |
| 全序列总投影代价 | $O(\sum_{t=1}^T t \cdot d^2) = O(___)$ | $O(___)$ |
| 第 $t$ 步：Attention 计算代价 | $O(t \cdot d)$ | $O(___)$ |

1. 解释标题"O(seq²) → O(seq)"指的是上表哪一行的代价
2. KV-Cache 节省了**投影**代价，Attention 计算代价是否也节省了？为什么？

（在此处填表和作答...）

In [ ]:
# 验证：timing 实验（3 次中位数，含 warmup，减少 flakiness）
import numpy as np, time

np.random.seed(0)
d, T_MAX = 128, 80

def _run_no_cache(xs, Wk, Wv):
    for t in range(1, len(xs) + 1):
        _ = xs[:t] @ Wk
        _ = xs[:t] @ Wv

def _run_cached(xs, Wk, Wv):
    K_cache, V_cache = [], []
    for x in xs:
        K_cache.append(x @ Wk)
        V_cache.append(x @ Wv)

def measure(fn, *args, reps=3):
    # warmup
    fn(*args)
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(*args)
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

xs = np.random.randn(T_MAX, d)
Wk = np.random.randn(d, d) * 0.02
Wv = np.random.randn(d, d) * 0.02

t_no  = measure(_run_no_cache, xs, Wk, Wv)
t_yes = measure(_run_cached,   xs, Wk, Wv)
speedup = t_no / t_yes

print(f"无缓存（中位数）: {t_no*1000:.2f} ms")
print(f"有缓存（中位数）: {t_yes*1000:.2f} ms")
print(f"加速比:          {speedup:.1f}×  (T={T_MAX}, d={d})")

assert speedup > 1.5, (
    f"有缓存应显著更快（期望 > 1.5×），当前 {speedup:.1f}×\n"
    f"若在高负载机器上偶发失败，可增大 T_MAX 或 reps 参数"
)
print(f"\n✅ KV-Cache 加速验证通过（{speedup:.1f}×）")

---

→ **下一课**　[L86 · 采样策略从零实现](L86_sampling.ipynb)

> 下节课将学习 **采样策略从零实现**：temperature / top-k / top-p，纯 NumPy。